# SEED Experiments

### Dependencies

In [1]:
import os
import random

In [2]:
import warnings
warnings.filterwarnings("ignore")

In [3]:
import re
import math
import torch
import joblib
import numpy as np
import pandas as pd
from tqdm import tqdm
import torch.nn as nn

import seaborn as sns
from sklearn.metrics import *
import torch.nn.functional as F
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib as mpl
from matplotlib.patches import Patch
from transformers import AutoModel
from datetime import datetime, timedelta
from transformers import get_cosine_schedule_with_warmup
from sklearn.preprocessing import MinMaxScaler
from torch.utils.data import Dataset, DataLoader
from transformers import BertModel, DistilBertModel
from transformers import BertTokenizer, DistilBertTokenizer, BertTokenizerFast
from transformers import AutoTokenizer, AutoModelForMaskedLM
from transformers import BertModel
from torch.cuda.amp import autocast, GradScaler
from transformers import GPT2Model

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [4]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

In [5]:
LOG_DIR   = '/kaggle/working/logs'
os.makedirs(LOG_DIR, exist_ok=True)

# Data Merging and Exploration

### New Data

In [6]:
# new cell
df = pd.read_csv("/kaggle/input/datasets/mhooshmand/t-mmd-us/US_FLURATIO_Week.csv")
df.head(3)

,date,start_date,end_date,REGION TYPE,REGION,YEAR,WEEK,% WEIGHTED ILI,OT,AGE 0-4,...,ILITOTAL,NUM. OF PROVIDERS,TOTAL PATIENTS,YEAR_WEEK,prior_history_avg,prior_history_std,Final_Search_2,Final_Search_4,Final_Search_6,Final_Output
0,9/29/1997,9/29/1997,10/5/1997,National,X,1997,40,1.10148,1.21686,179,...,570,192,46842,1997-40,0.0,0.0,Available facts are as follows: 1997-09-22: Za...,Available facts are as follows: 1997-09-22: Za...,Available facts are as follows: 1997-09-22: Za...,"Based on the provided textual information, I p..."
1,10/6/1997,10/6/1997,10/12/1997,National,X,1997,41,1.20007,1.28064,199,...,615,191,48023,1997-41,0.0,0.0,Available facts are as follows: 1997-09-29: Th...,Available facts are as follows: 1997-09-29: Th...,Available facts are as follows: 1997-09-29: Th...,"Based on the provided textual information, my ..."
2,10/13/1997,10/13/1997,10/19/1997,National,X,1997,42,1.37876,1.23906,228,...,681,219,54961,1997-42,0.0,0.0,Available facts are as follows: 1997-10-06: Th...,Available facts are as follows: 1997-10-06: Th...,Available facts are as follows: 1997-10-06: Th...,"Based on the available textual information, I ..."


In [7]:
# we will only keep the finaL_search_2 
text_df = pd.DataFrame()
text_df["facts"] = df["Final_Search_6"]
text_df.head()

,facts
0,Available facts are as follows: 1997-09-22: Za...
1,Available facts are as follows: 1997-09-29: Th...
2,Available facts are as follows: 1997-10-06: Th...
3,Available facts are as follows: 1997-10-13: Ob...
4,Available facts are as follows: 1997-10-20: Th...


In [8]:
text_df["preds"] = df["Final_Output"]
text_df.head(3)

,facts,preds
0,Available facts are as follows: 1997-09-22: Za...,"Based on the provided textual information, I p..."
1,Available facts are as follows: 1997-09-29: Th...,"Based on the provided textual information, my ..."
2,Available facts are as follows: 1997-10-06: Th...,"Based on the available textual information, I ..."


In [9]:
text_df.shape

(1389, 2)

In [10]:
numeric_df = pd.DataFrame()
numeric_df = df.iloc[:, :-4]
numeric_df.head(3)

,date,start_date,end_date,REGION TYPE,REGION,YEAR,WEEK,% WEIGHTED ILI,OT,AGE 0-4,...,AGE 25-64,AGE 5-24,AGE 50-64,AGE 65,ILITOTAL,NUM. OF PROVIDERS,TOTAL PATIENTS,YEAR_WEEK,prior_history_avg,prior_history_std
0,9/29/1997,9/29/1997,10/5/1997,National,X,1997,40,1.10148,1.21686,179,...,157,205,X,29,570,192,46842,1997-40,0.0,0.0
1,10/6/1997,10/6/1997,10/12/1997,National,X,1997,41,1.20007,1.28064,199,...,151,242,X,23,615,191,48023,1997-41,0.0,0.0
2,10/13/1997,10/13/1997,10/19/1997,National,X,1997,42,1.37876,1.23906,228,...,153,266,X,34,681,219,54961,1997-42,0.0,0.0


In [11]:
del df

In [12]:
numeric_df.drop(["date"], axis=1, inplace=True)
numeric_df['start_date'] = pd.to_datetime(numeric_df['start_date'])
numeric_df['end_date'] = pd.to_datetime(numeric_df['end_date'])
numeric_df.head(3)

,start_date,end_date,REGION TYPE,REGION,YEAR,WEEK,% WEIGHTED ILI,OT,AGE 0-4,AGE 25-49,AGE 25-64,AGE 5-24,AGE 50-64,AGE 65,ILITOTAL,NUM. OF PROVIDERS,TOTAL PATIENTS,YEAR_WEEK,prior_history_avg,prior_history_std
0,1997-09-29,1997-10-05,National,X,1997,40,1.10148,1.21686,179,X,157,205,X,29,570,192,46842,1997-40,0.0,0.0
1,1997-10-06,1997-10-12,National,X,1997,41,1.20007,1.28064,199,X,151,242,X,23,615,191,48023,1997-41,0.0,0.0
2,1997-10-13,1997-10-19,National,X,1997,42,1.37876,1.23906,228,X,153,266,X,34,681,219,54961,1997-42,0.0,0.0


In [13]:
text_df.head(3)

,facts,preds
0,Available facts are as follows: 1997-09-22: Za...,"Based on the provided textual information, I p..."
1,Available facts are as follows: 1997-09-29: Th...,"Based on the provided textual information, my ..."
2,Available facts are as follows: 1997-10-06: Th...,"Based on the available textual information, I ..."


In [14]:
print(f"text_df shape: {text_df.shape}\nnumeric_df shape: {numeric_df.shape}")

text_df shape: (1389, 2)
numeric_df shape: (1389, 20)


In [15]:
merged_df = pd.concat([numeric_df, text_df], axis=1)
merged_df.head(3)

,start_date,end_date,REGION TYPE,REGION,YEAR,WEEK,% WEIGHTED ILI,OT,AGE 0-4,AGE 25-49,...,AGE 50-64,AGE 65,ILITOTAL,NUM. OF PROVIDERS,TOTAL PATIENTS,YEAR_WEEK,prior_history_avg,prior_history_std,facts,preds
0,1997-09-29,1997-10-05,National,X,1997,40,1.10148,1.21686,179,X,...,X,29,570,192,46842,1997-40,0.0,0.0,Available facts are as follows: 1997-09-22: Za...,"Based on the provided textual information, I p..."
1,1997-10-06,1997-10-12,National,X,1997,41,1.20007,1.28064,199,X,...,X,23,615,191,48023,1997-41,0.0,0.0,Available facts are as follows: 1997-09-29: Th...,"Based on the provided textual information, my ..."
2,1997-10-13,1997-10-19,National,X,1997,42,1.37876,1.23906,228,X,...,X,34,681,219,54961,1997-42,0.0,0.0,Available facts are as follows: 1997-10-06: Th...,"Based on the available textual information, I ..."


In [16]:
merged_df.isnull().sum()

start_date            0
end_date              0
REGION TYPE           0
REGION                0
YEAR                  0
WEEK                  0
% WEIGHTED ILI        0
OT                    0
AGE 0-4               0
AGE 25-49             0
AGE 25-64             0
AGE 5-24              0
AGE 50-64             0
AGE 65                0
ILITOTAL              0
NUM. OF PROVIDERS     0
TOTAL PATIENTS        0
YEAR_WEEK             0
prior_history_avg     0
prior_history_std    53
facts                 0
preds                 0
dtype: int64

In [17]:
merged_df.isna().sum()

start_date            0
end_date              0
REGION TYPE           0
REGION                0
YEAR                  0
WEEK                  0
% WEIGHTED ILI        0
OT                    0
AGE 0-4               0
AGE 25-49             0
AGE 25-64             0
AGE 5-24              0
AGE 50-64             0
AGE 65                0
ILITOTAL              0
NUM. OF PROVIDERS     0
TOTAL PATIENTS        0
YEAR_WEEK             0
prior_history_avg     0
prior_history_std    53
facts                 0
preds                 0
dtype: int64

In [18]:
# Update ----- "Week" feature should be defined in a manner that reflects its cyclic nature
merged_df["WEEK_sin"] = np.sin(2 * np.pi * (merged_df["WEEK"]/52))
merged_df["WEEK_cos"] = np.cos(2 * np.pi * (merged_df["WEEK"]/52))

merged_df.drop(["WEEK"], axis=1, inplace=True)
merged_df.head(3)

,start_date,end_date,REGION TYPE,REGION,YEAR,% WEIGHTED ILI,OT,AGE 0-4,AGE 25-49,AGE 25-64,...,ILITOTAL,NUM. OF PROVIDERS,TOTAL PATIENTS,YEAR_WEEK,prior_history_avg,prior_history_std,facts,preds,WEEK_sin,WEEK_cos
0,1997-09-29,1997-10-05,National,X,1997,1.10148,1.21686,179,X,157,...,570,192,46842,1997-40,0.0,0.0,Available facts are as follows: 1997-09-22: Za...,"Based on the provided textual information, I p...",-0.992709,0.120537
1,1997-10-06,1997-10-12,National,X,1997,1.20007,1.28064,199,X,151,...,615,191,48023,1997-41,0.0,0.0,Available facts are as follows: 1997-09-29: Th...,"Based on the provided textual information, my ...",-0.970942,0.239316
2,1997-10-13,1997-10-19,National,X,1997,1.37876,1.23906,228,X,153,...,681,219,54961,1997-42,0.0,0.0,Available facts are as follows: 1997-10-06: Th...,"Based on the available textual information, I ...",-0.935016,0.354605


In [19]:
merged_df["REGION TYPE"].value_counts()

REGION TYPE
National    1389
Name: count, dtype: int64

In [20]:
merged_df["REGION"].value_counts()

REGION
X    1389
Name: count, dtype: int64

In [21]:
merged_df["YEAR_WEEK"].value_counts()

YEAR_WEEK
2024-19    1
1997-40    1
1997-41    1
1997-42    1
1997-43    1
          ..
1998-3     1
1998-4     1
1998-5     1
1998-6     1
1998-7     1
Name: count, Length: 1389, dtype: int64

In [22]:
merged_df.drop(["REGION TYPE", "REGION", "YEAR_WEEK"], axis=1, inplace=True)

In [23]:
merged_df.columns

Index(['start_date', 'end_date', 'YEAR', '% WEIGHTED ILI', 'OT', 'AGE 0-4',
       'AGE 25-49', 'AGE 25-64', 'AGE 5-24', 'AGE 50-64', 'AGE 65', 'ILITOTAL',
       'NUM. OF PROVIDERS', 'TOTAL PATIENTS', 'prior_history_avg',
       'prior_history_std', 'facts', 'preds', 'WEEK_sin', 'WEEK_cos'],
      dtype='object')

In [24]:
merged_df.isna().sum()

start_date            0
end_date              0
YEAR                  0
% WEIGHTED ILI        0
OT                    0
AGE 0-4               0
AGE 25-49             0
AGE 25-64             0
AGE 5-24              0
AGE 50-64             0
AGE 65                0
ILITOTAL              0
NUM. OF PROVIDERS     0
TOTAL PATIENTS        0
prior_history_avg     0
prior_history_std    53
facts                 0
preds                 0
WEEK_sin              0
WEEK_cos              0
dtype: int64

In [25]:
merged_df.drop(["prior_history_std", "prior_history_avg"], axis=1, inplace=True)
merged_df.isna().sum()

start_date           0
end_date             0
YEAR                 0
% WEIGHTED ILI       0
OT                   0
AGE 0-4              0
AGE 25-49            0
AGE 25-64            0
AGE 5-24             0
AGE 50-64            0
AGE 65               0
ILITOTAL             0
NUM. OF PROVIDERS    0
TOTAL PATIENTS       0
facts                0
preds                0
WEEK_sin             0
WEEK_cos             0
dtype: int64

In [26]:
cols = merged_df.columns
cols

Index(['start_date', 'end_date', 'YEAR', '% WEIGHTED ILI', 'OT', 'AGE 0-4',
       'AGE 25-49', 'AGE 25-64', 'AGE 5-24', 'AGE 50-64', 'AGE 65', 'ILITOTAL',
       'NUM. OF PROVIDERS', 'TOTAL PATIENTS', 'facts', 'preds', 'WEEK_sin',
       'WEEK_cos'],
      dtype='object')

In [27]:
# Updated ----------------------
# start_date, end_date, REGTION_TYPE
num_features = ['YEAR',
                "WEEK_sin",
                "WEEK_cos",
                '% WEIGHTED ILI',
                'AGE 0-4',
                'AGE 5-24',
                'AGE 25-49',
                'AGE 25-64',
                'AGE 50-64',
                'AGE 65',
                'ILITOTAL',
                'NUM. OF PROVIDERS',
                'TOTAL PATIENTS'
               ]

print(f"Number of features: {len(num_features)}") # should be 13
print(num_features)


Number of features: 13
['YEAR', 'WEEK_sin', 'WEEK_cos', '% WEIGHTED ILI', 'AGE 0-4', 'AGE 5-24', 'AGE 25-49', 'AGE 25-64', 'AGE 50-64', 'AGE 65', 'ILITOTAL', 'NUM. OF PROVIDERS', 'TOTAL PATIENTS']


In [28]:
merged_df.isna().sum()

start_date           0
end_date             0
YEAR                 0
% WEIGHTED ILI       0
OT                   0
AGE 0-4              0
AGE 25-49            0
AGE 25-64            0
AGE 5-24             0
AGE 50-64            0
AGE 65               0
ILITOTAL             0
NUM. OF PROVIDERS    0
TOTAL PATIENTS       0
facts                0
preds                0
WEEK_sin             0
WEEK_cos             0
dtype: int64

In [29]:
# age_cols = ['AGE 25-49', 'AGE 50-64', 'AGE 25-64'] 
# now do the below code for all columns, not just those columns mentioned above
cols = merged_df.columns
merged_df[cols] = merged_df[cols].replace('X', np.nan) # for all columns
print(merged_df[cols].isna().sum())

start_date             0
end_date               0
YEAR                   0
% WEIGHTED ILI         0
OT                     0
AGE 0-4                0
AGE 25-49            532
AGE 25-64            762
AGE 5-24               0
AGE 50-64            532
AGE 65                 0
ILITOTAL               0
NUM. OF PROVIDERS      0
TOTAL PATIENTS         0
facts                  0
preds                  0
WEEK_sin               0
WEEK_cos               0
dtype: int64


In [30]:
merged_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1389 entries, 0 to 1388
Data columns (total 18 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   start_date         1389 non-null   datetime64[ns]
 1   end_date           1389 non-null   datetime64[ns]
 2   YEAR               1389 non-null   int64         
 3   % WEIGHTED ILI     1389 non-null   float64       
 4   OT                 1389 non-null   float64       
 5   AGE 0-4            1389 non-null   int64         
 6   AGE 25-49          857 non-null    object        
 7   AGE 25-64          627 non-null    object        
 8   AGE 5-24           1389 non-null   int64         
 9   AGE 50-64          857 non-null    object        
 10  AGE 65             1389 non-null   int64         
 11  ILITOTAL           1389 non-null   int64         
 12  NUM. OF PROVIDERS  1389 non-null   int64         
 13  TOTAL PATIENTS     1389 non-null   int64         
 14  facts   

In [31]:
merged_df[num_features] = merged_df[num_features].apply(pd.to_numeric, errors='coerce') # just the numeric ones
merged_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1389 entries, 0 to 1388
Data columns (total 18 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   start_date         1389 non-null   datetime64[ns]
 1   end_date           1389 non-null   datetime64[ns]
 2   YEAR               1389 non-null   int64         
 3   % WEIGHTED ILI     1389 non-null   float64       
 4   OT                 1389 non-null   float64       
 5   AGE 0-4            1389 non-null   int64         
 6   AGE 25-49          857 non-null    float64       
 7   AGE 25-64          627 non-null    float64       
 8   AGE 5-24           1389 non-null   int64         
 9   AGE 50-64          857 non-null    float64       
 10  AGE 65             1389 non-null   int64         
 11  ILITOTAL           1389 non-null   int64         
 12  NUM. OF PROVIDERS  1389 non-null   int64         
 13  TOTAL PATIENTS     1389 non-null   int64         
 14  facts   

# Data Preprocessing

In [32]:
merged_df.sort_values('start_date', inplace=True)

In [33]:
# Chronological split: 80% train, 10% val, 10% test
train_size = int(len(merged_df) * 0.8)
val_size = int(len(merged_df) * 0.1)

train_df = merged_df.iloc[:train_size]
val_df = merged_df.iloc[train_size:train_size + val_size]
test_df = merged_df.iloc[train_size + val_size:]

print(f'Train: {len(train_df)*100/len(merged_df)}\nVal: {len(val_df)*100/len(merged_df)}\nTest: {len(test_df)*100/len(merged_df)}')

Train: 79.98560115190784
Val: 9.935205183585314
Test: 10.079193664506839


In [34]:
# Same Logic as before, but on scale of data frames
for df in [train_df, val_df, test_df]:
    mask_split_nan = df['AGE 25-49'].isna() & df['AGE 50-64'].isna() & ~df['AGE 25-64'].isna()
    df.loc[mask_split_nan, 'AGE 25-49'] = df.loc[mask_split_nan, 'AGE 25-64'] * 0.6
    df.loc[mask_split_nan, 'AGE 50-64'] = df.loc[mask_split_nan, 'AGE 25-64'] * 0.4
    
    mask_combined_nan = ~df['AGE 25-49'].isna() & ~df['AGE 50-64'].isna() & df['AGE 25-64'].isna()
    df.loc[mask_combined_nan, 'AGE 25-64'] = df.loc[mask_combined_nan, 'AGE 25-49'] + df.loc[mask_combined_nan, 'AGE 50-64']

In [35]:
print(merged_df.isna().sum())

start_date           0
end_date             0
YEAR                 0
% WEIGHTED ILI       0
OT                   0
AGE 0-4              0
AGE 25-49            0
AGE 25-64            0
AGE 5-24             0
AGE 50-64            0
AGE 65               0
ILITOTAL             0
NUM. OF PROVIDERS    0
TOTAL PATIENTS       0
facts                0
preds                0
WEEK_sin             0
WEEK_cos             0
dtype: int64


In [36]:
merged_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1389 entries, 0 to 1388
Data columns (total 18 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   start_date         1389 non-null   datetime64[ns]
 1   end_date           1389 non-null   datetime64[ns]
 2   YEAR               1389 non-null   int64         
 3   % WEIGHTED ILI     1389 non-null   float64       
 4   OT                 1389 non-null   float64       
 5   AGE 0-4            1389 non-null   int64         
 6   AGE 25-49          1389 non-null   float64       
 7   AGE 25-64          1389 non-null   float64       
 8   AGE 5-24           1389 non-null   int64         
 9   AGE 50-64          1389 non-null   float64       
 10  AGE 65             1389 non-null   int64         
 11  ILITOTAL           1389 non-null   int64         
 12  NUM. OF PROVIDERS  1389 non-null   int64         
 13  TOTAL PATIENTS     1389 non-null   int64         
 14  facts   

In [37]:
num_features

['YEAR',
 'WEEK_sin',
 'WEEK_cos',
 '% WEIGHTED ILI',
 'AGE 0-4',
 'AGE 5-24',
 'AGE 25-49',
 'AGE 25-64',
 'AGE 50-64',
 'AGE 65',
 'ILITOTAL',
 'NUM. OF PROVIDERS',
 'TOTAL PATIENTS']

In [38]:
scaler = MinMaxScaler()
train_df[num_features] = scaler.fit_transform(train_df[num_features])
val_df[num_features] = scaler.transform(val_df[num_features])
test_df[num_features] = scaler.transform(test_df[num_features])

# Saved for later (inverse transform predictions)
joblib.dump(scaler, '/kaggle/working/feature_scaler.pkl')


# scaling targets (Updated to prevent leakage)
target_scaler = MinMaxScaler()
train_df["OT"] = target_scaler.fit_transform(train_df["OT"].values.reshape(-1,1)).flatten()
val_df["OT"] = target_scaler.transform(val_df["OT"].values.reshape(-1,1)).flatten()
test_df["OT"] = target_scaler.transform(test_df["OT"].values.reshape(-1,1)).flatten()

joblib.dump(target_scaler, '/kaggle/working/target_scaler.pkl')

# Updated
num_features.append("OT")

In [39]:
print(f"numerical features:\n{num_features}")

numerical features:
['YEAR', 'WEEK_sin', 'WEEK_cos', '% WEIGHTED ILI', 'AGE 0-4', 'AGE 5-24', 'AGE 25-49', 'AGE 25-64', 'AGE 50-64', 'AGE 65', 'ILITOTAL', 'NUM. OF PROVIDERS', 'TOTAL PATIENTS', 'OT']


In [40]:
print(f"columns:{merged_df.columns}\nshape:{merged_df.shape}")

columns:Index(['start_date', 'end_date', 'YEAR', '% WEIGHTED ILI', 'OT', 'AGE 0-4',
       'AGE 25-49', 'AGE 25-64', 'AGE 5-24', 'AGE 50-64', 'AGE 65', 'ILITOTAL',
       'NUM. OF PROVIDERS', 'TOTAL PATIENTS', 'facts', 'preds', 'WEEK_sin',
       'WEEK_cos'],
      dtype='object')
shape:(1389, 18)


In [41]:
# Same functionality as before but with Horizon & Look Back

def create_sequences(df, lookback, horizon):
    df = df.sort_values('start_date').reset_index(drop=True)  # Ensure order
    X_num, X_text, y=[], [], []
        
    for i in range(len(df) - lookback - horizon + 1):
        # 1. get the numerical features
        num_seq = df[num_features].iloc[i:i+lookback].values # (36,14)
        
        # 2. textuals too
        # Updated: Changing the texual contents into a list of values
        # text_seq = (df['facts'].iloc[i:i+lookback] + ' ' + df['preds'].iloc[i:i+lookback]).to_list()  # update: we no longer need the facts
        # text_seq = (df['preds'].iloc[i:i+lookback] + ' ' + df['facts'].iloc[i:i+lookback]).to_list() # to make sure facts survives the textual stream
        # text_seq = (df['preds'].iloc[i:i+lookback]).to_list()
        text_seq = (df['facts'].iloc[i:i+lookback]).to_list()
        
        # 3. and the OT (target)
        target = df['OT'].iloc[i+lookback:i+lookback+horizon].values # (12,)

        X_num.append(num_seq)
        X_text.append(text_seq)
        y.append(target)
        
    return np.array(X_num), np.array(X_text), np.array(y)
        

In [42]:
# Train / Val / Test splitting
lookback = 36 # Default of Time-MMD
horizon = 12  # predict 12 weeks at once

# X_num_train, X_text_train, y_train = create_sequences(train_df, lookback, horizon)
# X_num_val, X_text_val, y_val = create_sequences(val_df, lookback, horizon)
# X_num_test, X_text_test, y_test = create_sequences(test_df, lookback, horizon)

# Update --------
# Prepend the tail of the previous split so the first sequence 
# in val/test has a full 36-week lookback drawn from real history
val_df_full = pd.concat([train_df.iloc[-lookback:], val_df],  ignore_index=True)
test_df_full = pd.concat([val_df.iloc[-lookback:],  test_df], ignore_index=True)

X_num_train, X_text_train, y_train = create_sequences(train_df, lookback, horizon)
X_num_val, X_text_val, y_val = create_sequences(val_df_full, lookback, horizon)
X_num_test, X_text_test, y_test = create_sequences(test_df_full, lookback, horizon)

In [43]:
# Double checking
X_num_train.shape

(1064, 36, 14)

In [44]:
X_text_train.shape

(1064, 36)

In [45]:
# Double checking
y_train.shape

(1064, 12)

In [46]:

def tokenize_headlines(sequences, max_len=256, tokenizer=None): 
    """Added the functionality to handle various tokenizers and also made the caching mechanism faster"""
    if tokenizer is None:
        tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')  
    
    tokenized = []
    for sample in tqdm(sequences): 
        sample = sample.tolist()
        # sample: list of 36 weeks       
        enc = tokenizer(
            sample,                      # batch of 36 strings
            max_length = max_len,
            padding = 'max_length',
            truncation = True,
            return_tensors = 'pt'
            )

        tokenized.append({
            'input_ids': enc['input_ids'],
            'attention_mask': enc['attention_mask']
        })
        
    return tokenized 
  

## Dataset Creation

In [47]:
X_text_train.shape

(1064, 36)

In [48]:
X_text_train_tokenized = tokenize_headlines(X_text_train, 
                                            # tokenizer=DistilBertTokenizer.from_pretrained("distilbert-base-uncased")
                                            # tokenizer = tokenizer,
                                            # tokenizer = AutoTokenizer.from_pretrained("microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext")
                                            tokenizer = AutoTokenizer.from_pretrained("michiyasunaga/BioLinkBERT-base")
                                           )

X_text_val_tokenized = tokenize_headlines(X_text_val, 
                                          # tokenizer=DistilBertTokenizer.from_pretrained("distilbert-base-uncased")
                                          # tokenizer = tokenizer
                                          # tokenizer = AutoTokenizer.from_pretrained("microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext")
                                          tokenizer = AutoTokenizer.from_pretrained("michiyasunaga/BioLinkBERT-base")
                                         )

X_text_test_tokenized = tokenize_headlines(X_text_test, 
                                           # tokenizer=DistilBertTokenizer.from_pretrained("distilbert-base-uncased")
                                           # tokenizer = tokenizer
                                           # tokenizer = AutoTokenizer.from_pretrained("microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext")
                                           tokenizer = AutoTokenizer.from_pretrained("michiyasunaga/BioLinkBERT-base")
                                          )

config.json:   0%|          | 0.00/559 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/379 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

100%|██████████| 129/129 [00:06<00:00, 19.19it/s]


In [49]:
class CustomDataset(Dataset):
    """Modified to have a 3D shape output"""
    def __init__(self, numerical_data, tokenized_text_data, targets):
        self.numerical_data = torch.tensor(numerical_data, dtype=torch.float32)          
        self.tokenized_text_data = tokenized_text_data  # List of dicts
        self.targets = torch.tensor(targets, dtype=torch.float32) 


    def __len__(self):
        return len(self.numerical_data)


    def __getitem__(self, idx):
        num = self.numerical_data[idx]  # (36, 14)
        text_input = self.tokenized_text_data[idx]  # Dict: (36, max_len)
        target = self.targets[idx]  # (12,)
        return num, text_input, target



In [50]:
train_dataset = CustomDataset(X_num_train, 
                              X_text_train_tokenized, 
                              y_train)

val_dataset = CustomDataset(X_num_val, 
                            X_text_val_tokenized, 
                            y_val)

test_dataset = CustomDataset(X_num_test, 
                             X_text_test_tokenized,
                             y_test)

In [51]:
batch_size = 4
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)  
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

# Neural Network Components — GPT4MTS (LLM-based multimodal baseline)

**Why GPT4MTS over ChatTime:** both were flagged as candidate LLM-based multimodal baselines. GPT4MTS (Jia et al., AAAI 2024) builds on the FPT/GPT4TS ('One Fits All') convention: a **frozen, 6-layer GPT-2 backbone** (~an order of magnitude smaller than even GPT-2's full 12 layers), with only positional embeddings, LayerNorm, and the task head trained -- deliberately lightweight by design. ChatTime (AAAI 2025) is a **7B-parameter LLaMA-2-based foundation model**; even with LoRA + 4-bit quantization, fine-tuning a 7B backbone ten times (one per seed) is a very different time budget than fine-tuning a 6-layer, mostly-frozen GPT-2. Given the explicit goal of a full 10-seed sweep, GPT4MTS is the only realistic choice on a 2xT4 Kaggle budget.

**Fidelity notes:**
1. **Text encoder**: frozen BioLinkBERT-base -- identical checkpoint and masked-mean-pooling to DSA's and TaTS's text encoders, so the text *representation* is held constant across every multimodal method compared in this study. GPT4MTS's own design uses BERT purely as an off-the-shelf feature extractor for its textual 'prompt', which this matches.
2. **GPT-2(6) backbone**: first 6 of GPT-2's 12 pretrained transformer blocks; attention and feed-forward sublayers frozen; positional embeddings, both LayerNorms per block, and the final LayerNorm are fine-tuned -- this is the standard FPT/GPT4TS freezing scheme that GPT4MTS itself builds on.
3. **RevIN is OFF by default** (`use_revin=False`), overriding GPT4MTS's own default, for the same reason it's disabled in the TaTS backbones elsewhere in this study: this project already established RevIN-style instance normalization fails on this non-stationary epidemic data.
4. **Prompt mechanism**: the exact token-level mechanics of GPT4MTS's 'prompt layer' are not fully specified in the paper/public materials at the level of detail needed to guarantee a byte-for-byte reproduction. The implementation below is a faithful-in-spirit reconstruction: pooled textual context for the whole lookback window is projected into a small number of learned 'soft prompt' tokens, prepended to the (channel-independent, PatchTST-style) numerical patch tokens before being fed into GPT-2(6). If you have access to the official GPT4MTS repository and want an exact reproduction instead, flag it and this can be re-wired around that implementation directly, the same way the TaTS backbones were re-wired around your actual naive-fusion notebooks.

## Frozen Text Encoder (identical to TaTS's, for a fair comparison)

In [52]:
class GPT4MTSTextEncoder(nn.Module):
    """Frozen BioLinkBERT-base, used purely as a feature extractor per
    GPT4MTS's own design. Identical checkpoint and masked-mean-pooling to
    DSA's HeadlineEncoder and TaTS's TaTSTextEncoder."""
    def __init__(self):
        super().__init__()
        self.bert = AutoModel.from_pretrained("michiyasunaga/BioLinkBERT-base")
        for p in self.bert.parameters():
            p.requires_grad = False
        self.bert.eval()

    @torch.no_grad()
    def forward(self, input_ids, attention_masks):
        B, T, L = input_ids.shape
        flat_ids = input_ids.view(-1, L)
        flat_mask = attention_masks.view(-1, L)

        out = self.bert(input_ids=flat_ids, attention_mask=flat_mask)
        last_hidden = out.last_hidden_state                       # (B*T, L, 768)

        token_mask = flat_mask.unsqueeze(-1)
        pooled = (last_hidden * token_mask).sum(dim=1) / token_mask.sum(dim=1).clamp(min=1e-6)

        return pooled.view(B, T, 768)

## Prompt Projector (text -> soft-prompt tokens in GPT-2's input space)

In [53]:
class PromptProjector(nn.Module):
    """Converts the pooled textual context for the whole lookback window
    into n_prompt_tokens learned 'soft prompt' embeddings, matching
    GPT4MTS's prompt-layer role of injecting textual context into GPT-2's
    input space alongside the numerical patch tokens."""
    def __init__(self, d_text=768, d_model=768, n_prompt_tokens=4, dropout=0.15):
        super().__init__()
        self.n_prompt_tokens = n_prompt_tokens
        self.d_model = d_model
        self.proj = nn.Sequential(
            nn.Linear(d_text, d_model),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model, d_model * n_prompt_tokens)
        )

    def forward(self, text_repr):           # (B, T, d_text)
        pooled = text_repr.mean(dim=1)        # (B, d_text) -- summarize context over the whole window
        out = self.proj(pooled)                # (B, d_model * n_prompt_tokens)
        return out.view(-1, self.n_prompt_tokens, self.d_model)

## Patch Embedding (channel-independent, PatchTST-style, directly into GPT-2's hidden size)

In [54]:
class PatchEmbedding(nn.Module):
    """Channel-independent patching of the numerical input, projected
    directly into GPT-2's hidden size (768). RevIN is included but
    disabled by default -- see fidelity note 3 above."""
    def __init__(self, seq_len, patch_len=4, stride=2, d_model=768, use_revin=False):
        super().__init__()
        self.patch_len = patch_len
        self.stride = stride
        self.use_revin = use_revin
        self.num_patches = (seq_len - patch_len) // stride + 1
        self.patch_proj = nn.Linear(patch_len, d_model)
        self.pos_emb = nn.Parameter(torch.zeros(1, self.num_patches, d_model))
        nn.init.trunc_normal_(self.pos_emb, std=0.02)

    def forward(self, x):                     # x: (B, T, C)
        B, T, C = x.shape

        if self.use_revin:
            means = x.mean(1, keepdim=True).detach()
            x = x - means
            stds = torch.sqrt(x.var(1, keepdim=True, unbiased=False) + 1e-5)
            x = x / stds

        x = x.transpose(1, 2)                                       # (B, C, T)
        x = x.unfold(dimension=2, size=self.patch_len, step=self.stride)  # (B, C, num_patches, patch_len)
        x = x.reshape(B * C, self.num_patches, self.patch_len)

        x = self.patch_proj(x) + self.pos_emb                        # (B*C, num_patches, d_model)
        return x

## Frozen GPT-2(6) Backbone (FPT / GPT4TS 'One Fits All' convention)

In [55]:
class FrozenGPT2Backbone(nn.Module):
    """First 6 of GPT-2's 12 pretrained transformer blocks. Attention and
    feed-forward sublayers are frozen; positional embeddings and both
    LayerNorms per block (plus the final LayerNorm) are fine-tuned. This
    is the standard FPT/GPT4TS freezing scheme -- GPT4MTS itself builds
    on this exact convention, adding the multimodal prompt fusion on top."""
    def __init__(self, n_layers=6):
        super().__init__()
        self.gpt2 = GPT2Model.from_pretrained('gpt2')
        self.gpt2.h = self.gpt2.h[:n_layers]
        self.gpt2.config.n_layer = n_layers

        for name, param in self.gpt2.named_parameters():
            if name.startswith('wte'):
                param.requires_grad = False           # token embedding -- unused (we pass inputs_embeds directly)
            elif name.startswith('wpe'):
                param.requires_grad = True             # positional embedding -- fine-tuned
            elif '.attn.' in name or '.mlp.' in name:
                param.requires_grad = False            # attention + feed-forward -- frozen (retain pretrained knowledge)
            elif '.ln_1.' in name or '.ln_2.' in name or name.startswith('ln_f'):
                param.requires_grad = True             # LayerNorm -- fine-tuned (domain adaptation)
            else:
                param.requires_grad = False            # safety catch-all

    def forward(self, inputs_embeds):
        out = self.gpt2(inputs_embeds=inputs_embeds)
        return out.last_hidden_state

# Proposed Framework Wrapper

In [56]:
class GPT4MTS(nn.Module):
    def __init__(self, seq_len, horizon, num_channels, text_encoder,
                 patch_len=4, stride=2, n_prompt_tokens=4, d_model=768,
                 gpt2_layers=6, dropout=0.15, use_revin=False):
        super().__init__()
        self.text_encoder = text_encoder
        self.prompt_projector = PromptProjector(768, d_model, n_prompt_tokens, dropout)
        self.patch_embed = PatchEmbedding(seq_len, patch_len, stride, d_model, use_revin)
        self.gpt2 = FrozenGPT2Backbone(gpt2_layers)
        self.n_prompt_tokens = n_prompt_tokens
        self.num_channels = num_channels
        self.dropout = nn.Dropout(dropout)
        self.head = nn.Sequential(
            nn.Linear(self.patch_embed.num_patches * d_model, d_model),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model, horizon)
        )

    def forward(self, input_ids, attention_masks, numerical_features):
        B, T, C = numerical_features.shape

        text_repr = self.text_encoder(input_ids, attention_masks)          # (B, T, 768)
        prompt_tokens = self.prompt_projector(text_repr)                    # (B, n_prompt, 768)

        patch_tokens = self.patch_embed(numerical_features)                  # (B*C, num_patches, 768)

        # Broadcast the same textual prompt across every channel's patch
        # sequence (channels stay independent per PatchTST/GPT4TS convention;
        # the shared prompt is the only cross-channel information injected).
        prompt_tokens_expanded = prompt_tokens.unsqueeze(1).expand(-1, C, -1, -1)
        prompt_tokens_expanded = prompt_tokens_expanded.reshape(B * C, self.n_prompt_tokens, -1)

        combined = torch.cat([prompt_tokens_expanded, patch_tokens], dim=1)    # (B*C, n_prompt+num_patches, 768)
        combined = self.dropout(combined)

        gpt2_out = self.gpt2(inputs_embeds=combined)                           # (B*C, n_prompt+num_patches, 768)
        patch_out = gpt2_out[:, self.n_prompt_tokens:, :]                        # discard prompt positions

        patch_out = patch_out.reshape(B * C, -1)
        out = self.head(patch_out)                                              # (B*C, horizon)
        out = out.reshape(B, C, -1).permute(0, 2, 1)                             # (B, horizon, C)
        return out


class GPT4MTSForecaster(nn.Module):
    """Thin wrapper slicing out the target channel, matching the same
    (B, horizon, C) -> (B, horizon) contract used by the TaTS backbones."""
    def __init__(self, gpt4mts_model, target_idx):
        super().__init__()
        self.model = gpt4mts_model
        self.target_idx = target_idx

    def forward(self, input_ids, attention_masks, numerical_features):
        out = self.model(input_ids, attention_masks, numerical_features)
        return out[:, :, self.target_idx]

# Training Loop

## Loop Init

**SEED is the only thing you change between runs.** Following the same workflow used for DSA/iTransformer/TaTS elsewhere in this project: set `SEED` below, run the notebook top to bottom on Kaggle, record the printed Test MSE/MAE/RMSE, then repeat with the next seed. Artifacts are saved per-seed with `GPT4MTS` in the filename so they slot directly into your existing DM-test / tail-risk analysis scripts.

In [57]:

epochs = 150
patience = 10
warm_up_epochs = 10
accumulation_steps = 4     # effective batch = 16, matching DSA/TaTS
n_prompt_tokens = 8 # update
use_revin = True # update
gpt2_layers = 8

target_idx = num_features.index("OT")
print(f"SEED={SEED} | target_idx (within {len(num_features)} channels): {target_idx}")

SEED=42 | target_idx (within 14 channels): 13


## Core

In [58]:
text_encoder = GPT4MTSTextEncoder().to(device)
gpt4mts_model = GPT4MTS(
    seq_len=lookback,
    horizon=horizon,
    num_channels=len(num_features),
    text_encoder=text_encoder,
    patch_len=8, #update
    stride=2,
    n_prompt_tokens=n_prompt_tokens,
    d_model=768,
    gpt2_layers=gpt2_layers,
    dropout=0.20, #update
    use_revin=use_revin
).to(device)

model = GPT4MTSForecaster(gpt4mts_model, target_idx).to(device)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable params: {trainable:,} / {total:,} ({100*trainable/total:.1f}%)")

mse = nn.MSELoss()
optimizer = torch.optim.AdamW([
    {'params': [p for n, p in model.named_parameters() if p.requires_grad and 'gpt2' not in n], 'lr': 1e-3},
    {'params': [p for n, p in model.named_parameters() if p.requires_grad and 'gpt2' in n],     'lr': 1e-4},
], weight_decay=1e-4)

amp_scaler = GradScaler()
updates_per_epoch = math.ceil(len(train_loader) / accumulation_steps)
scheduler = get_cosine_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warm_up_epochs * updates_per_epoch,
    num_training_steps=epochs * updates_per_epoch
)

history = {'epoch': [], 'train_loss': [], 'val_loss': []}
best_val_loss = float('inf')
p_counter = 0
best_model_path = os.path.join(LOG_DIR, f'GPT4MTS_seed{SEED}_best.pth')

for epoch in tqdm(range(epochs)):
    model.train()
    epoch_train_loss = 0.0
    optimizer.zero_grad()

    for batch_idx, batch in enumerate(train_loader):
        num = batch[0].to(device)
        text_input = {k: v.to(device) for k, v in batch[1].items()}
        targets = batch[2].to(device)

        with autocast():
            preds = model(text_input['input_ids'], text_input['attention_mask'], num)
            loss = mse(preds, targets) / accumulation_steps

        amp_scaler.scale(loss).backward()

        if (batch_idx + 1) % accumulation_steps == 0 or (batch_idx + 1) == len(train_loader):
            amp_scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            amp_scaler.step(optimizer)
            amp_scaler.update()
            scheduler.step()
            optimizer.zero_grad()

        epoch_train_loss += loss.item() * accumulation_steps

    avg_train_loss = epoch_train_loss / len(train_loader)

    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for batch in val_loader:
            num = batch[0].to(device)
            text_input = {k: v.to(device) for k, v in batch[1].items()}
            targets = batch[2].to(device)
            with autocast():
                preds = model(text_input["input_ids"], text_input['attention_mask'], num)
                loss = mse(preds, targets)
            val_loss += loss.item()
    avg_val_loss = val_loss / len(val_loader)

    history['epoch'].append(epoch)
    history['train_loss'].append(avg_train_loss)
    history['val_loss'].append(avg_val_loss)

    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        p_counter = 0
        torch.save(model.state_dict(), best_model_path)
    else:
        if epoch >= warm_up_epochs:
            p_counter += 1
            if p_counter >= patience:
                print(f"Early stopping at epoch {epoch}")
                break

    print(f"[SEED {SEED}] Epoch {epoch} | Train Loss: {avg_train_loss:.6f} | Val Loss: {avg_val_loss:.6f}")

pytorch_model.bin:   0%|          | 0.00/433M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: michiyasunaga/BioLinkBERT-base
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


model.safetensors:   0%|          | 0.00/433M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

...

[SEED 42] Epoch 25 | Train Loss: 0.009921 | Val Loss: 0.023493


 18%|█▊        | 27/150 [1:22:02<6:13:41, 182.29s/it]

[SEED 42] Epoch 26 | Train Loss: 0.010694 | Val Loss: 0.027524


 18%|█▊        | 27/150 [1:25:05<6:27:36, 189.08s/it]

Early stopping at epoch 27


## Test Evaluation (identical metric pipeline to DSA/TaTS)

In [59]:
model.load_state_dict(torch.load(best_model_path, map_location=device))
model.eval()
test_preds, test_targets = [], []
with torch.no_grad():
    for batch in test_loader:
        num = batch[0].to(device)
        text_input = {k: v.to(device) for k, v in batch[1].items()}
        targets = batch[2].to(device)
        preds = model(text_input['input_ids'], text_input['attention_mask'], num)
        test_preds.extend(preds.cpu().numpy())
        test_targets.extend(targets.cpu().numpy())

test_preds = np.array(test_preds)
test_targets = np.array(test_targets)

inv_preds = target_scaler.inverse_transform(test_preds.reshape(-1, 1)).reshape(-1, horizon)
inv_targets = target_scaler.inverse_transform(test_targets.reshape(-1, 1)).reshape(-1, horizon)

mse_val = mean_squared_error(inv_targets.flatten(), inv_preds.flatten())
mae_val = mean_absolute_error(inv_targets.flatten(), inv_preds.flatten())
rmse_val = np.sqrt(mse_val)

per_sample_mse = ((inv_preds - inv_targets) ** 2).mean(axis=1)
per_horizon_mse = ((inv_preds - inv_targets) ** 2).mean(axis=0)

np.save(f'{LOG_DIR}/errors_GPT4MTS_{SEED}.npy', per_sample_mse)
np.save(f'{LOG_DIR}/horizon_GPT4MTS_{SEED}.npy', per_horizon_mse)
np.save(f'{LOG_DIR}/preds_GPT4MTS_{SEED}.npy', inv_preds)
np.save(f'{LOG_DIR}/targets_GPT4MTS_{SEED}.npy', inv_targets)

print(f"\nGPT4MTS (SEED={SEED})  |  Test MSE: {mse_val:.4f}  MAE: {mae_val:.4f}  RMSE: {rmse_val:.4f}")


GPT4MTS (SEED=42)  |  Test MSE: 1.3400  MAE: 0.8128  RMSE: 1.1576


In [60]:
# !rm -rf /kaggle/working/*